# 958 - Recovering eBFE Archives That `zipfile` Refuses

FEMA publishes eBFE deliveries as very large ZIP archives. Two in the Texas corpus
**cannot be opened by Python's `zipfile` at all**:

| Study | Archive | Problem |
|---|---|---|
| `12100302` Medina | `Medina_Models.zip`, 52.09 GB | Local headers declare 53.02 GB. The last member runs 935,101,777 bytes past the end of the file, and the central directory - which would have followed it - was never written. |
| `12110205` Baffin Bay | `Input.zip`, 68.6 GB | 73% zero padding, corrupt final member, no central directory. |

Both match their published `Content-Length` and ETag exactly, so **the truncation is at
the publisher** - re-downloading does not help.

`zipfile` finds members by seeking to the End Of Central Directory record at the tail of
the file and reading the index it points at. With no such record there is no index, and
`ZipFile()` raises `BadZipFile` before you can read a single member.

`StreamingZipReader` walks **local file headers forward** instead. It needs no index and
no tail, so it recovers everything actually present. On Medina that was 407 of 407
recoverable members, all CRC-verified.

## What you'll learn

- Why a missing central directory breaks random access, and what to use instead
- Surveying an archive with `probe()` without reading member data
- Detecting truncation explicitly rather than mistaking it for a clean end-of-file
- Extracting selectively with `want`, so probe and extract are one traversal
- Verifying with the ZIP format's own CRC-32 instead of hashing

## Prerequisites

ras-commander and a HEC-RAS example project via `RasExamples`. No HEC-RAS install
required. Runtime under a minute.


In [ ]:
USE_LOCAL_SOURCE = True

if USE_LOCAL_SOURCE:
    import sys
    from pathlib import Path
    local_path = str(Path.cwd().parent)
    if local_path not in sys.path:
        sys.path.insert(0, local_path)

import shutil
import zipfile
from pathlib import Path

from ras_commander import RasExamples
from ras_commander.sources.federal.ebfe_extract import StreamingZipReader

print('supported compression methods:', StreamingZipReader.supported_methods())


## Building a realistic archive

Rather than synthesise bytes, package a **real HEC-RAS project** - the kind of content
an eBFE delivery carries - then damage a copy the way the publisher did.


In [ ]:
project = RasExamples.extract_project('Muncie', suffix='zipdemo')
work = Path('example_projects') / 'streaming_zip_demo'
work.mkdir(parents=True, exist_ok=True)

intact = work / 'MuncieDelivery.zip'
with zipfile.ZipFile(intact, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(project.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(project).as_posix())

print(f'{intact.name}: {intact.stat().st_size / 1e6:.1f} MB')


## Example 1 - an intact archive

`probe()` reads **headers only**. It never touches member data, so surveying a 52 GB
archive costs a header walk rather than a full read.


In [ ]:
reader = StreamingZipReader(intact)
survey = reader.probe()

print(f'members               : {len(survey.members)}')
print(f'central directory     : {survey.has_central_directory}')
print(f'truncated             : {survey.truncated}')
print(f'stopped because       : {survey.stopped_reason}')
print(f'projected uncompressed: {survey.projected_bytes / 1e6:.1f} MB')


### Selective extraction

`want` is evaluated **before** any member data is read, so pulling a handful of files
from a huge archive does not inflate the rest. That is why probe and extract are the
same traversal rather than two passes.

`sink_factory` returns a writable handle per member. Taking a factory rather than a
destination directory keeps path policy - flattening, Windows `MAX_PATH` projection -
with the caller, where it belongs.


In [ ]:
geometry_out = work / 'geometry_only'

def make_sink(member):
    target = geometry_out / member.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return open(target, 'wb')

reader = StreamingZipReader(intact)
wanted = [
    member.name
    for member, took in reader.walk(
        want=lambda m: m.name.endswith(('.prj', '.g01', '.p01')),
        sink_factory=make_sink,
    )
    if took
]

print('extracted:', wanted)
print(f'skipped  : {reader.stats.skipped}')
print(f'bytes read from archive: {reader.stats.bytes_read:,}')
print(f'archive size           : {intact.stat().st_size:,}')
print(f'CRC verified: {reader.stats.crc_ok}, failures: {reader.stats.crc_fail}')


Note the byte count - only the wanted members were inflated.

**CRC is the verification, not a hash.** The ZIP format stores a CRC-32 per member, so
checking it costs nothing beyond the decompression already happening, and proves
byte-correctness without a second full read. Hashing 16 TB of corpus would buy nothing
this does not already provide.


## Example 2 - the Medina failure mode

Now damage a copy the way the publisher did: cut the file mid-member, taking the
central directory and the tail with it.


In [ ]:
truncated = work / 'MuncieDelivery_truncated.zip'
raw = intact.read_bytes()
truncated.write_bytes(raw[: int(len(raw) * 0.6)])

try:
    zipfile.ZipFile(truncated)
    print('zipfile opened it')
except zipfile.BadZipFile as exc:
    print(f'zipfile refuses it: {exc}')


In [ ]:
reader = StreamingZipReader(truncated)
survey = reader.probe()

print(f'truncated       : {survey.truncated}')
print(f'headers declare : {survey.declared_end:,} bytes')
print(f'file holds      : {survey.file_size:,} bytes')
print(f'short by        : {survey.overrun_bytes:,} bytes')
print(f'recoverable     : {len(survey.complete_members)} of {len(survey.members)}')
print(f'stopped because : {survey.stopped_reason}')


> **Why `stopped_reason` matters.** Seeking past the end of a file does not raise - the
> next read simply returns empty bytes. Without an explicit check the walk reports a tidy
> `eof` while sitting hundreds of megabytes beyond the end, and a truncated delivery looks
> exactly like a healthy one. The reader compares the declared end against the real file
> size and reports `truncated` instead.


In [ ]:
recovered_out = work / 'recovered'

def recover_sink(member):
    target = recovered_out / member.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return open(target, 'wb')

reader = StreamingZipReader(truncated)
results = list(reader.walk(sink_factory=recover_sink))

print(f'recovered : {reader.stats.extracted}')
print(f'unreadable: {reader.stats.unreadable}')
print(f'CRC ok    : {reader.stats.crc_ok}, CRC failures: {reader.stats.crc_fail}')
print(f'truncated : {reader.stats.truncated}  (never read, so never a CRC failure)')
print()
for name, kind, reason in reader.stats.failures[:5]:
    print(f'  [{kind}] {name}: {reason}')


Every member wholly present is recovered and CRC-verified. Members whose data runs past
the end are reported as **unreadable with a reason**, not silently written short - a
partially-written HEC-RAS geometry file that looks complete is far worse than one that is
explicitly missing.


## Key takeaways

- **A missing central directory is recoverable.** Walk local headers forward; the member
  data is still there.
- **This is not a `ZipFile` drop-in, deliberately.** Random access is precisely what a
  missing central directory cannot offer, and an API implying otherwise invites
  `namelist()`-then-`read()` code that reads the archive twice.
- **Check truncation explicitly.** A seek past EOF is silent.
- **`projected_bytes` is an estimate**, not a guarantee - members with deferred sizes
  report zero - so leave margin when using it as a disk-space gate.
- **deflate64 (method 9) appears in real deliveries** and the standard library cannot read
  it. Install the optional `zipfile-deflate64` package;
  `StreamingZipReader.supported_methods()` reports what is live.

## Adapting this

```python
reader = StreamingZipReader(archive_path)
survey = reader.probe()
if survey.truncated:
    log_gap(archive_path, survey.overrun_bytes, survey.truncated_members)

for member, took in reader.walk(want=my_filter, sink_factory=my_sink):
    ...

assert reader.stats.crc_fail == 0, reader.stats.failures
```


In [ ]:
shutil.rmtree(work, ignore_errors=True)
print('cleaned up')
